The code below grabs cells until it identifies an area that contains a median cell at least 95% of the time. However, this code performed this function when I was previously grabbing the first median cell identified for each county. I need to account for the fact that there will be multiple daily median cells. This will likely involve summing the number of each cell in order to calculate the 95% threshold 


#### Identify 95% Coverage for Median Alternative PM2.5 Value Locations in Each Contiguous U.S. County



In [0]:
#Purpose of Cell Block: Grab each counties median cells until we encompass an area that is the median for the county at least 95% of the time in 2019

#Count the number of times a given cell was a median cell
county_cell_freq <- results_all_days %>%
    filter(!is.na(median_cell_value)) %>%
    count(
        county_row,
        cell,
        x,
        y,
        name = "n_days_at_cell"
    )

#Count total number of days of data in each county 
county_total_days <- results_all_days %>%
    filter(!is.na(median_cell_value)) %>%
    count(
        county_row,
        name = "total_days"
    )

coverage_target <- 0.95 #95% of the median cells will fall within a given area (identifying the region where the median cell typicall falls)

#Most frequent cells per county, grabbing only the amount i need to obtain goal coverage
county_multi_cell <- county_cell_freq %>%
    left_join(county_total_days, by = "county_row") %>% #join frequency datafram with the total number of days per county datafram for percentagecalculation
    mutate(
        pct_days = n_days_at_cell / total_days #caluclate the percentage of days that a cell was the median county cell
    ) %>%
    arrange(county_row, desc(n_days_at_cell)) %>% #arrange by county in decsending number of median cell days 
    group_by(county_row) %>% #for each county... 
    mutate(
        cumul_pct_days = cumsum(pct_days) #calculate the cumulative sum of a numeric vector (pct_days)
    ) %>%
    filter(cumul_pct_days <= coverage_target | lag(cumul_pct_days, default = 0) < coverage_target) %>% #only show the cumsum that is just greater than the target
    ungroup()


head(county_multi_cell)

write_csv(
    county_multi_cell,
    "D_processed_data/intermediate_scratch/county_multi_cell.csv"
)

#Conclusion: The dataset was successfully generated

In [0]:
#Purpose of Cell Block: Verify the dataset contains expected values

setwd("/home/user/capstone/A_data")
contiguous_counties <- vect("D_processed_data/intermediate_scratch/contiguous_counties_geom.shp")

county_multi_cell <- read.csv("D_processed_data/intermediate_scratch/county_multi_cell.csv")

head(county_multi_cell)

# Diagnostics for county_multi_cell

glimpse(county_multi_cell)

required_cols <- c("county_row","cell","x","y","n_days_at_cell","total_days","pct_days","cumul_pct_days")
missing_cols <- setdiff(required_cols, names(county_multi_cell))
stopifnot(length(missing_cols) == 0)

# Check whether there are NAs in the dataset
na_counts <- county_multi_cell %>%
  summarise(across(all_of(required_cols), ~sum(is.na(.)), .names = "na_{.col}"))
na_counts

# Check the ranges for each of the columns
range_checks <- county_multi_cell %>%
  summarise(
    n_rows = n(),
    county_min = min(county_row, na.rm = TRUE),
    county_max = max(county_row, na.rm = TRUE),
    n_days_min = min(n_days_at_cell, na.rm = TRUE),
    n_days_max = max(n_days_at_cell, na.rm = TRUE),
    total_days_min = min(total_days, na.rm = TRUE),
    total_days_max = max(total_days, na.rm = TRUE),
    pct_days_min = min(pct_days, na.rm = TRUE),
    pct_days_max = max(pct_days, na.rm = TRUE),
    cumul_min = min(cumul_pct_days, na.rm = TRUE),
    cumul_max = max(cumul_pct_days, na.rm = TRUE)
  )
range_checks

# Confirm no logical violations in dataset (e.g., a percent value greater than 100%)
logic_violations <- county_multi_cell %>%
  mutate(
    bad_counts = (n_days_at_cell < 0) | (total_days <= 0) | (n_days_at_cell > total_days), #There should be no counts less than zero and number of days at cellsould be less than total days
    bad_pct = (pct_days < 0) | (pct_days > 1), #Percent should not be less than 0 or greater than 1
    bad_cumul = (cumul_pct_days < 0) | (cumul_pct_days > 1), #Same logic as line above
    bad_geo = (x < -180 | x > 180 | y < -90 | y > 90)  # checking if unusual xy coords 
  ) %>%
  summarise( #create new rows to sum violations
    n_bad_counts = sum(bad_counts, na.rm = TRUE),
    n_bad_pct = sum(bad_pct, na.rm = TRUE),
    n_bad_cumul = sum(bad_cumul, na.rm = TRUE),
    n_bad_geo = sum(bad_geo, na.rm = TRUE)
  )
logic_violations

# Overall distributions
county_multi_cell %>%
  summarise(
    n_rows = n(),
    n_counties = n_distinct(county_row),
    n_cells = n_distinct(cell)
  )


ggplot(county_multi_cell, aes(x = pct_days)) + #Distribution of pct_days
  geom_histogram(bins = 50) +
  labs(title = "Distribution of pct_days", x = "pct_days", y = "count")

#Conclsion: The dataset appears to be correct. Interestingly, it appears at least one county had 95% of its median cells in one location while another county's median cells spanned 138 cells. 